In [1]:
# ============================================================
# Robustness / Perturbation Experiments for
# Ablation-Driven Evaluation of Trivial Feature Baselines
#
# Focus:
# - FraudYelp
# - FraudAmazon
#
# Excluded:
# - PyG Reddit
# - DGraph-Fin
# - Bitcoin pseudo-label stress tests
#
# Saves:
# - one JSON per experiment
# - one TXT log per experiment
# - summary tables
# - robustness drop tables
#
# Resume-safe:
# - existing JSON runs are skipped
# ============================================================

from __future__ import annotations

import os
import sys
import json
import time
import math
import random
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    IsolationForest,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets")

OUT_ROOT = Path.cwd() / "robustness_attribute_structure_outputs"
LOG_DIR = OUT_ROOT / "logs"
JSON_DIR = OUT_ROOT / "results_json"
SUMMARY_DIR = OUT_ROOT / "summary"
CACHE_DIR = OUT_ROOT / "cached_perturbed_features"

for d in [OUT_ROOT, LOG_DIR, JSON_DIR, SUMMARY_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Working dir:", Path.cwd())
print("DATA_ROOT:", DATA_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("SEED:", SEED)

Python: D:\Users\ivan\anaconda3\envs\gad_dgl\python.exe
Working dir: C:\Users\ivan\WORK\workshops\ICDM\Ablation
DATA_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets
OUT_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\robustness_attribute_structure_outputs
SEED: 42


In [2]:
# ============================================================
# Dataset paths
# ============================================================

PATHS = {
    # FraudYelp / YelpChi
    "fraud_yelp_features": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_feature.pt",
    "fraud_yelp_labels": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_label.pt",
    "fraud_yelp_train_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_train_mask.pt",
    "fraud_yelp_val_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_val_mask.pt",
    "fraud_yelp_test_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_test_mask.pt",
    "fraud_yelp_edges_rsr": DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rsr_review" / "fraud_yelp_review_net_rsr_review_edges.csv",
    "fraud_yelp_edges_rtr": DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rtr_review" / "fraud_yelp_review_net_rtr_review_edges.csv",
    "fraud_yelp_edges_rur": DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rur_review" / "fraud_yelp_review_net_rur_review_edges.csv",

    # FraudAmazon
    "fraud_amazon_features": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_feature.pt",
    "fraud_amazon_labels": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_label.pt",
    "fraud_amazon_train_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_train_mask.pt",
    "fraud_amazon_val_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_val_mask.pt",
    "fraud_amazon_test_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_test_mask.pt",
    "fraud_amazon_edges_upu": DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_upu_user" / "fraud_amazon_user_net_upu_user_edges.csv",
    "fraud_amazon_edges_usu": DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_usu_user" / "fraud_amazon_user_net_usu_user_edges.csv",
    "fraud_amazon_edges_uvu": DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_uvu_user" / "fraud_amazon_user_net_uvu_user_edges.csv",
}

print("=" * 120)
print("PATH CHECK")
print("=" * 120)

for name, path in PATHS.items():
    status = "OK" if path.exists() else "MISSING"
    size = path.stat().st_size / 1024 / 1024 if path.exists() and path.is_file() else None
    if size is None:
        print(f"{status:<8} {name:<35} {path}")
    else:
        print(f"{status:<8} {name:<35} {path} | {size:.2f} MB")

PATH CHECK
OK       fraud_yelp_features                 C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_feature.pt | 5.61 MB
OK       fraud_yelp_labels                   C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_label.pt | 0.35 MB
OK       fraud_yelp_train_mask               C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_train_mask.pt | 0.04 MB
OK       fraud_yelp_val_mask                 C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_val_mask.pt | 0.04 MB
OK       fraud_yelp_test_mask                C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_test_mask.pt | 0.04 MB
OK       fraud_yelp_edges_rsr                

In [3]:
# ============================================================
# Logging and saving
# ============================================================

def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def safe_name(x: str) -> str:
    x = str(x)
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ', '(', ')', '[', ']', '{', '}', ',', ';']
    for b in bad:
        x = x.replace(b, "_")
    while "__" in x:
        x = x.replace("__", "_")
    return x.strip("_")


def log_line(log_path: Path, msg: str, also_print: bool = True):
    line = f"[{now_str()}] {msg}"
    if also_print:
        print(line)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def save_json(path: Path, obj: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    tmp.replace(path)


def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def tensor_to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def print_class_balance(y, prefix=""):
    y = np.asarray(y).astype(int)
    vals, counts = np.unique(y, return_counts=True)
    total = len(y)
    print(prefix + "class balance:")
    for v, c in zip(vals, counts):
        print(prefix + f"  class {v}: {c} ({c / total:.4f})")

In [4]:
# ============================================================
# Metrics
# ============================================================

def precision_recall_hits_at_k(
    y_true: np.ndarray,
    scores: np.ndarray,
    k: Optional[int] = None,
) -> Dict[str, float]:

    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)

    n = len(y_true)
    n_pos = int(y_true.sum())

    if k is None:
        k = max(1, n_pos)

    k = int(max(1, min(k, n)))

    order = np.argsort(-scores)
    top = order[:k]
    hits = int(y_true[top].sum())

    return {
        "k": k,
        "hits": hits,
        "precision_at_k": float(hits / k),
        "recall_at_k": float(hits / max(1, n_pos)),
        "hits_at_k": float(hits > 0),
    }


def compute_metrics(y_true: np.ndarray, scores: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)

    out = {}

    if len(np.unique(y_true)) < 2:
        out["roc_auc"] = None
        out["pr_auc"] = None
    else:
        out["roc_auc"] = float(roc_auc_score(y_true, scores))
        out["pr_auc"] = float(average_precision_score(y_true, scores))

    n = len(y_true)
    n_pos = int(y_true.sum())

    k_list = {
        "at_pos": max(1, n_pos),
        "at_1pct": max(1, int(0.01 * n)),
        "at_5pct": max(1, int(0.05 * n)),
        "at_10pct": max(1, int(0.10 * n)),
    }

    for tag, k in k_list.items():
        d = precision_recall_hits_at_k(y_true, scores, k=k)
        for key, value in d.items():
            out[f"{key}_{tag}"] = value

    return out

In [5]:
# ============================================================
# Edge loading and structural features
# ============================================================

def load_relation_edges(edge_paths: List[Path]) -> pd.DataFrame:
    dfs = []

    for p in edge_paths:
        if not p.exists():
            print("[WARN] missing edge file:", p)
            continue

        df = pd.read_csv(p)

        if "source" not in df.columns or "target" not in df.columns:
            raise ValueError(f"Edge file has no source/target columns: {p}")

        df = df[["source", "target"]].copy()
        df["weight"] = 1.0
        df["relation"] = p.parent.name
        dfs.append(df)

    if not dfs:
        raise RuntimeError("No edge files loaded.")

    return pd.concat(dfs, ignore_index=True)


def compute_structural_features_from_edges(
    edges: pd.DataFrame,
    n_nodes: int,
    node_ids: Optional[np.ndarray] = None,
    weight_col: str = "weight",
) -> Tuple[np.ndarray, List[str]]:

    if node_ids is None:
        node_ids = np.arange(n_nodes)

    node_to_idx = {int(node): int(i) for i, node in enumerate(node_ids)}

    src = edges["source"].map(node_to_idx)
    dst = edges["target"].map(node_to_idx)

    valid = src.notna() & dst.notna()
    src = src[valid].to_numpy(dtype=np.int64)
    dst = dst[valid].to_numpy(dtype=np.int64)

    if weight_col in edges.columns:
        w = edges.loc[valid, weight_col].to_numpy(dtype=float)
    else:
        w = np.ones(len(src), dtype=float)

    eps = 1e-9

    out_deg = np.bincount(src, minlength=n_nodes).astype(float)
    in_deg = np.bincount(dst, minlength=n_nodes).astype(float)
    total_deg = in_deg + out_deg

    out_w_sum = np.bincount(src, weights=w, minlength=n_nodes).astype(float)
    in_w_sum = np.bincount(dst, weights=w, minlength=n_nodes).astype(float)

    abs_w = np.abs(w)
    out_abs_w_sum = np.bincount(src, weights=abs_w, minlength=n_nodes).astype(float)
    in_abs_w_sum = np.bincount(dst, weights=abs_w, minlength=n_nodes).astype(float)

    pos = (w > 0).astype(float)
    neg = (w < 0).astype(float)

    out_pos = np.bincount(src, weights=pos, minlength=n_nodes).astype(float)
    in_pos = np.bincount(dst, weights=pos, minlength=n_nodes).astype(float)
    out_neg = np.bincount(src, weights=neg, minlength=n_nodes).astype(float)
    in_neg = np.bincount(dst, weights=neg, minlength=n_nodes).astype(float)

    in_neg_ratio = in_neg / (in_pos + in_neg + eps)
    out_neg_ratio = out_neg / (out_pos + out_neg + eps)
    all_neg_ratio = (in_neg + out_neg) / (in_pos + out_pos + in_neg + out_neg + eps)

    # Neighbor degree aggregation.
    neighbor_deg_sum_out = np.bincount(src, weights=total_deg[dst], minlength=n_nodes).astype(float)
    neighbor_deg_sum_in = np.bincount(dst, weights=total_deg[src], minlength=n_nodes).astype(float)

    avg_out_neighbor_deg = neighbor_deg_sum_out / (out_deg + eps)
    avg_in_neighbor_deg = neighbor_deg_sum_in / (in_deg + eps)
    avg_neighbor_deg = (neighbor_deg_sum_out + neighbor_deg_sum_in) / (total_deg + eps)

    # Reciprocity.
    # For huge graphs, set-based reciprocity can be expensive, but for these datasets it is acceptable.
    pair_set = set(zip(src.tolist(), dst.tolist()))
    reciprocal_flags = np.fromiter(
        ((b, a) in pair_set for a, b in zip(src, dst)),
        dtype=float,
        count=len(src),
    )

    out_recip = np.bincount(src, weights=reciprocal_flags, minlength=n_nodes).astype(float)
    in_recip = np.bincount(dst, weights=reciprocal_flags, minlength=n_nodes).astype(float)
    reciprocity_ratio = (out_recip + in_recip) / (total_deg + eps)

    # Relation-level diversity.
    if "relation" in edges.columns:
        relation_counts = []
        for rel in sorted(edges["relation"].dropna().unique()):
            rel_edges = edges[edges["relation"] == rel]
            rel_src = rel_edges["source"].map(node_to_idx)
            rel_dst = rel_edges["target"].map(node_to_idx)
            rel_valid = rel_src.notna() & rel_dst.notna()
            rel_src = rel_src[rel_valid].to_numpy(dtype=np.int64)
            rel_dst = rel_dst[rel_valid].to_numpy(dtype=np.int64)

            rel_out = np.bincount(rel_src, minlength=n_nodes).astype(float)
            rel_in = np.bincount(rel_dst, minlength=n_nodes).astype(float)
            relation_counts.append(rel_out + rel_in)

        if relation_counts:
            rel_mat = np.vstack(relation_counts).T
            rel_total = rel_mat.sum(axis=1) + eps
            rel_probs = rel_mat / rel_total[:, None]
            rel_entropy = -(rel_probs * np.log(rel_probs + eps)).sum(axis=1)
            rel_nonzero = (rel_mat > 0).sum(axis=1).astype(float)
        else:
            rel_entropy = np.zeros(n_nodes)
            rel_nonzero = np.zeros(n_nodes)
    else:
        rel_entropy = np.zeros(n_nodes)
        rel_nonzero = np.zeros(n_nodes)

    X = np.vstack([
        in_deg,
        out_deg,
        total_deg,
        np.log1p(in_deg),
        np.log1p(out_deg),
        np.log1p(total_deg),

        in_w_sum,
        out_w_sum,
        in_abs_w_sum,
        out_abs_w_sum,

        in_pos,
        out_pos,
        in_neg,
        out_neg,
        in_neg_ratio,
        out_neg_ratio,
        all_neg_ratio,

        avg_in_neighbor_deg,
        avg_out_neighbor_deg,
        avg_neighbor_deg,
        reciprocity_ratio,

        rel_entropy,
        rel_nonzero,
    ]).T.astype(np.float32)

    names = [
        "in_degree",
        "out_degree",
        "total_degree",
        "log1p_in_degree",
        "log1p_out_degree",
        "log1p_total_degree",

        "in_weight_sum",
        "out_weight_sum",
        "in_abs_weight_sum",
        "out_abs_weight_sum",

        "in_pos_count",
        "out_pos_count",
        "in_neg_count",
        "out_neg_count",
        "in_neg_ratio",
        "out_neg_ratio",
        "all_neg_ratio",

        "avg_in_neighbor_degree",
        "avg_out_neighbor_degree",
        "avg_neighbor_degree",
        "reciprocity_ratio",

        "relation_entropy",
        "relation_nonzero_count",
    ]

    return X, names

In [6]:
# ============================================================
# Perturbations
# ============================================================

def perturb_attributes(
    X: np.ndarray,
    regime: str,
    seed: int = SEED,
) -> np.ndarray:

    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=np.float32).copy()

    if regime == "original":
        return X

    if regime == "attr_shuffle":
        # Shuffle node-feature rows.
        perm = rng.permutation(X.shape[0])
        return X[perm].astype(np.float32)

    if regime == "attr_noise_10":
        std = np.nanstd(X, axis=0, keepdims=True)
        noise = rng.normal(0.0, 0.10, size=X.shape).astype(np.float32) * (std + 1e-6)
        return (X + noise).astype(np.float32)

    if regime == "attr_noise_25":
        std = np.nanstd(X, axis=0, keepdims=True)
        noise = rng.normal(0.0, 0.25, size=X.shape).astype(np.float32) * (std + 1e-6)
        return (X + noise).astype(np.float32)

    if regime == "attr_dropout_25":
        mask = rng.random(X.shape[1]) >= 0.25
        if mask.sum() == 0:
            mask[rng.integers(0, X.shape[1])] = True
        X2 = X.copy()
        X2[:, ~mask] = 0.0
        return X2.astype(np.float32)

    if regime == "attr_dropout_50":
        mask = rng.random(X.shape[1]) >= 0.50
        if mask.sum() == 0:
            mask[rng.integers(0, X.shape[1])] = True
        X2 = X.copy()
        X2[:, ~mask] = 0.0
        return X2.astype(np.float32)

    if regime in [
        "relation_dropout_25",
        "relation_dropout_50",
        "degree_preserving_rewire",
        "edges_removed",
    ]:
        # Attribute side unchanged.
        return X

    raise ValueError(f"Unknown attribute perturbation regime: {regime}")


def perturb_edges(
    edges: pd.DataFrame,
    regime: str,
    n_nodes: int,
    seed: int = SEED,
) -> pd.DataFrame:

    rng = np.random.default_rng(seed)
    edges = edges.copy()

    if regime in [
        "original",
        "attr_shuffle",
        "attr_noise_10",
        "attr_noise_25",
        "attr_dropout_25",
        "attr_dropout_50",
    ]:
        return edges

    if regime == "edges_removed":
        return pd.DataFrame({
            "source": np.array([], dtype=int),
            "target": np.array([], dtype=int),
            "weight": np.array([], dtype=float),
            "relation": np.array([], dtype=object),
        })

    if regime == "relation_dropout_25":
        frac = 0.25
        keep = rng.random(len(edges)) >= frac
        return edges.loc[keep].reset_index(drop=True)

    if regime == "relation_dropout_50":
        frac = 0.50
        keep = rng.random(len(edges)) >= frac
        return edges.loc[keep].reset_index(drop=True)

    if regime == "degree_preserving_rewire":
        # Directed degree-preserving-ish rewiring:
        # preserve source multiset and target multiset by shuffling target endpoints.
        # This preserves out-degree and in-degree distributions,
        # while destroying local neighborhoods and relation-specific structure.
        out = edges.copy()
        targets = out["target"].to_numpy().copy()
        rng.shuffle(targets)
        out["target"] = targets

        # Keep relation labels as they were; this tests whether local connection pattern matters.
        return out.reset_index(drop=True)

    raise ValueError(f"Unknown edge perturbation regime: {regime}")

In [7]:
# ============================================================
# Dataset loader
# ============================================================

def load_fraud_dataset(
    name: str,
    feature_path: Path,
    label_path: Path,
    train_mask_path: Path,
    val_mask_path: Path,
    test_mask_path: Path,
    edge_paths: List[Path],
) -> Dict[str, Any]:

    log_path = LOG_DIR / f"load_{safe_name(name)}.txt"
    log_line(log_path, f"Loading dataset: {name}")

    X_attr = torch.load(feature_path, map_location="cpu")
    y = torch.load(label_path, map_location="cpu")
    train_mask = torch.load(train_mask_path, map_location="cpu")
    val_mask = torch.load(val_mask_path, map_location="cpu")
    test_mask = torch.load(test_mask_path, map_location="cpu")

    X_attr = tensor_to_numpy(X_attr).astype(np.float32)
    y = tensor_to_numpy(y).astype(int).reshape(-1)
    train_mask = tensor_to_numpy(train_mask).astype(bool).reshape(-1)
    val_mask = tensor_to_numpy(val_mask).astype(bool).reshape(-1)
    test_mask = tensor_to_numpy(test_mask).astype(bool).reshape(-1)

    edges = load_relation_edges(edge_paths)

    n_nodes = len(y)

    train_idx = np.where(train_mask)[0]
    val_idx = np.where(val_mask)[0]
    test_idx = np.where(test_mask)[0]

    if len(train_idx) == 0 or len(test_idx) == 0:
        raise RuntimeError(f"{name}: bad train/test masks.")

    log_line(log_path, f"nodes={n_nodes}")
    log_line(log_path, f"edges={len(edges)}")
    log_line(log_path, f"features={X_attr.shape}")
    log_line(log_path, f"anomalies={int(y.sum())}")
    log_line(log_path, f"train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
    print_class_balance(y, prefix=f"{name} ")

    return {
        "name": name,
        "X_attr": X_attr,
        "y": y,
        "edges": edges,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "attr_names": [f"attr_{i}" for i in range(X_attr.shape[1])],
    }


datasets = []

datasets.append(load_fraud_dataset(
    name="FraudYelp",
    feature_path=PATHS["fraud_yelp_features"],
    label_path=PATHS["fraud_yelp_labels"],
    train_mask_path=PATHS["fraud_yelp_train_mask"],
    val_mask_path=PATHS["fraud_yelp_val_mask"],
    test_mask_path=PATHS["fraud_yelp_test_mask"],
    edge_paths=[
        PATHS["fraud_yelp_edges_rsr"],
        PATHS["fraud_yelp_edges_rtr"],
        PATHS["fraud_yelp_edges_rur"],
    ],
))

datasets.append(load_fraud_dataset(
    name="FraudAmazon",
    feature_path=PATHS["fraud_amazon_features"],
    label_path=PATHS["fraud_amazon_labels"],
    train_mask_path=PATHS["fraud_amazon_train_mask"],
    val_mask_path=PATHS["fraud_amazon_val_mask"],
    test_mask_path=PATHS["fraud_amazon_test_mask"],
    edge_paths=[
        PATHS["fraud_amazon_edges_upu"],
        PATHS["fraud_amazon_edges_usu"],
        PATHS["fraud_amazon_edges_uvu"],
    ],
))

print("=" * 120)
print("LOADED DATASETS")
print("=" * 120)

for ds in datasets:
    print(
        ds["name"],
        "nodes=", len(ds["y"]),
        "edges=", len(ds["edges"]),
        "features=", ds["X_attr"].shape,
        "anomalies=", int(ds["y"].sum()),
    )

[2026-05-17 14:06:58] Loading dataset: FraudYelp
[2026-05-17 14:07:05] nodes=45954
[2026-05-17 14:07:05] edges=8051348
[2026-05-17 14:07:05] features=(45954, 32)
[2026-05-17 14:07:05] anomalies=6677
[2026-05-17 14:07:05] train=32167, val=4595, test=9192
FraudYelp class balance:
FraudYelp   class 0: 39277 (0.8547)
FraudYelp   class 1: 6677 (0.1453)
[2026-05-17 14:07:05] Loading dataset: FraudAmazon
[2026-05-17 14:07:11] nodes=11944
[2026-05-17 14:07:11] edges=9557648
[2026-05-17 14:07:11] features=(11944, 25)
[2026-05-17 14:07:11] anomalies=821
[2026-05-17 14:07:11] train=6047, val=863, test=1729
FraudAmazon class balance:
FraudAmazon   class 0: 11123 (0.9313)
FraudAmazon   class 1: 821 (0.0687)
LOADED DATASETS
FraudYelp nodes= 45954 edges= 8051348 features= (45954, 32) anomalies= 6677
FraudAmazon nodes= 11944 edges= 9557648 features= (11944, 25) anomalies= 821


In [8]:
# ============================================================
# Build and cache perturbed feature sets
# ============================================================

PERTURBATION_REGIMES = [
    "original",
    "attr_shuffle",
    "attr_noise_10",
    "attr_noise_25",
    "attr_dropout_25",
    "attr_dropout_50",
    "relation_dropout_25",
    "relation_dropout_50",
    "degree_preserving_rewire",
    "edges_removed",
]

def cache_path_for_features(dataset_name: str, regime: str) -> Path:
    return CACHE_DIR / f"{safe_name(dataset_name)}__{safe_name(regime)}.npz"


def build_perturbed_dataset_features(
    ds: Dict[str, Any],
    regime: str,
    force: bool = False,
) -> Dict[str, Any]:

    dataset_name = ds["name"]
    path = cache_path_for_features(dataset_name, regime)
    log_path = LOG_DIR / f"feature_build_{safe_name(dataset_name)}__{safe_name(regime)}.txt"

    if path.exists() and not force:
        log_line(log_path, f"[SKIP] cached features exist: {path}")
        data = np.load(path, allow_pickle=True)
        return {
            "X_attr": data["X_attr"],
            "X_struct": data["X_struct"],
            "attr_names": data["attr_names"].tolist(),
            "struct_names": data["struct_names"].tolist(),
            "n_edges": int(data["n_edges"]),
        }

    log_line(log_path, "=" * 100)
    log_line(log_path, f"Building perturbed features: dataset={dataset_name}, regime={regime}")
    log_line(log_path, "=" * 100)

    t0 = time.time()

    X_attr_original = ds["X_attr"]
    edges_original = ds["edges"]
    n_nodes = len(ds["y"])

    X_attr_pert = perturb_attributes(
        X=X_attr_original,
        regime=regime,
        seed=SEED,
    )

    edges_pert = perturb_edges(
        edges=edges_original,
        regime=regime,
        n_nodes=n_nodes,
        seed=SEED,
    )

    if len(edges_pert) == 0:
        X_struct = np.zeros((n_nodes, 23), dtype=np.float32)
        struct_names = [
            "in_degree",
            "out_degree",
            "total_degree",
            "log1p_in_degree",
            "log1p_out_degree",
            "log1p_total_degree",
            "in_weight_sum",
            "out_weight_sum",
            "in_abs_weight_sum",
            "out_abs_weight_sum",
            "in_pos_count",
            "out_pos_count",
            "in_neg_count",
            "out_neg_count",
            "in_neg_ratio",
            "out_neg_ratio",
            "all_neg_ratio",
            "avg_in_neighbor_degree",
            "avg_out_neighbor_degree",
            "avg_neighbor_degree",
            "reciprocity_ratio",
            "relation_entropy",
            "relation_nonzero_count",
        ]
    else:
        X_struct, struct_names = compute_structural_features_from_edges(
            edges=edges_pert,
            n_nodes=n_nodes,
            node_ids=np.arange(n_nodes),
            weight_col="weight",
        )

    attr_names = ds["attr_names"]

    np.savez_compressed(
        path,
        X_attr=X_attr_pert.astype(np.float32),
        X_struct=X_struct.astype(np.float32),
        attr_names=np.array(attr_names, dtype=object),
        struct_names=np.array(struct_names, dtype=object),
        n_edges=np.array(len(edges_pert), dtype=np.int64),
    )

    runtime = time.time() - t0

    log_line(log_path, f"X_attr shape: {X_attr_pert.shape}")
    log_line(log_path, f"X_struct shape: {X_struct.shape}")
    log_line(log_path, f"n_edges original: {len(edges_original)}")
    log_line(log_path, f"n_edges perturbed: {len(edges_pert)}")
    log_line(log_path, f"Saved: {path}")
    log_line(log_path, f"Runtime sec: {runtime:.2f}")

    return {
        "X_attr": X_attr_pert,
        "X_struct": X_struct,
        "attr_names": attr_names,
        "struct_names": struct_names,
        "n_edges": len(edges_pert),
    }


# Build all caches once. Resume-safe.
for ds in datasets:
    for regime in PERTURBATION_REGIMES:
        print("\n" + "=" * 120)
        print(f"BUILD FEATURES: dataset={ds['name']}, regime={regime}")
        print("=" * 120)
        _ = build_perturbed_dataset_features(ds, regime, force=False)


BUILD FEATURES: dataset=FraudYelp, regime=original
[2026-05-17 14:07:42] ====================================================================================================
[2026-05-17 14:07:42] Building perturbed features: dataset=FraudYelp, regime=original
[2026-05-17 14:07:42] ====================================================================================================
[2026-05-17 14:08:04] X_attr shape: (45954, 32)
[2026-05-17 14:08:04] X_struct shape: (45954, 23)
[2026-05-17 14:08:04] n_edges original: 8051348
[2026-05-17 14:08:04] n_edges perturbed: 8051348
[2026-05-17 14:08:04] Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\robustness_attribute_structure_outputs\cached_perturbed_features\FraudYelp__original.npz
[2026-05-17 14:08:04] Runtime sec: 22.31

BUILD FEATURES: dataset=FraudYelp, regime=attr_shuffle
[2026-05-17 14:08:04] ====================================================================================================
[2026-05-17 14:08:04] Building perturbed

In [9]:
# ============================================================
# Ablation feature matrices
# ============================================================

ABLATIONS = [
    "attr_only",
    "struct_only",
    "full",
    "degree_only",
    "no_degree_struct",
]

def get_ablation_matrix(
    pert: Dict[str, Any],
    ablation: str,
) -> Tuple[np.ndarray, List[str]]:

    X_attr = pert["X_attr"]
    X_struct = pert["X_struct"]
    attr_names = pert["attr_names"]
    struct_names = pert["struct_names"]

    if ablation == "attr_only":
        return X_attr.astype(np.float32), attr_names

    if ablation == "struct_only":
        return X_struct.astype(np.float32), struct_names

    if ablation == "full":
        X = np.concatenate([X_attr, X_struct], axis=1).astype(np.float32)
        names = list(attr_names) + list(struct_names)
        return X, names

    if ablation == "degree_only":
        cols = []
        names = []
        for i, n in enumerate(struct_names):
            if "degree" in n or "deg" in n:
                cols.append(i)
                names.append(n)

        if not cols:
            return np.zeros((X_struct.shape[0], 1), dtype=np.float32), ["dummy_no_degree"]

        return X_struct[:, cols].astype(np.float32), names

    if ablation == "no_degree_struct":
        cols = []
        names = []
        for i, n in enumerate(struct_names):
            if "degree" not in n and "deg" not in n:
                cols.append(i)
                names.append(n)

        if not cols:
            return np.zeros((X_struct.shape[0], 1), dtype=np.float32), ["dummy_no_non_degree"]

        return X_struct[:, cols].astype(np.float32), names

    raise ValueError(f"Unknown ablation: {ablation}")

print("Perturbation regimes:", PERTURBATION_REGIMES)
print("Ablations:", ABLATIONS)

Perturbation regimes: ['original', 'attr_shuffle', 'attr_noise_10', 'attr_noise_25', 'attr_dropout_25', 'attr_dropout_50', 'relation_dropout_25', 'relation_dropout_50', 'degree_preserving_rewire', 'edges_removed']
Ablations: ['attr_only', 'struct_only', 'full', 'degree_only', 'no_degree_struct']


In [10]:
# ============================================================
# Models
# ============================================================

def make_supervised_model(model_name: str):
    if model_name == "LogisticRegression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="lbfgs",
                random_state=SEED,
            )),
        ])

    if model_name == "RandomForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=SEED,
            )),
        ])

    if model_name == "ExtraTrees":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED,
            )),
        ])

    if model_name == "GradientBoosting":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", GradientBoostingClassifier(
                n_estimators=150,
                learning_rate=0.05,
                max_depth=3,
                random_state=SEED,
            )),
        ])

    raise ValueError(model_name)


def make_unsupervised_model(model_name: str, contamination: float):
    if model_name == "IsolationForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", IsolationForest(
                n_estimators=300,
                contamination=max(1e-4, min(0.49, contamination)),
                random_state=SEED,
                n_jobs=-1,
            )),
        ])

    raise ValueError(model_name)


SUPERVISED_MODELS = [
    "LogisticRegression",
    "RandomForest",
    "ExtraTrees",
    "GradientBoosting",
]

UNSUPERVISED_MODELS = [
    "IsolationForest",
]

ALL_MODELS = SUPERVISED_MODELS + UNSUPERVISED_MODELS

print("Models:", ALL_MODELS)

Models: ['LogisticRegression', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'IsolationForest']


In [11]:
# ============================================================
# Run one experiment
# ============================================================

def experiment_id(
    dataset_name: str,
    regime: str,
    ablation: str,
    model_name: str,
) -> str:
    return "__".join([
        safe_name(dataset_name),
        safe_name(regime),
        safe_name(ablation),
        safe_name(model_name),
    ])


def experiment_paths(
    dataset_name: str,
    regime: str,
    ablation: str,
    model_name: str,
) -> Tuple[Path, Path]:
    eid = experiment_id(dataset_name, regime, ablation, model_name)
    return JSON_DIR / f"{eid}.json", LOG_DIR / f"{eid}.txt"


def fit_predict_scores(
    model_name: str,
    X: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> Tuple[np.ndarray, Dict[str, Any]]:

    y = y.astype(int)

    X_train = X[train_idx]
    y_train = y[train_idx]
    X_test = X[test_idx]

    info = {}

    if model_name in SUPERVISED_MODELS:
        model = make_supervised_model(model_name)

        if len(np.unique(y_train)) < 2:
            raise RuntimeError("Only one class in training data.")

        model.fit(X_train, y_train)

        clf = model.named_steps["clf"]

        if hasattr(clf, "predict_proba"):
            scores = model.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            scores = model.decision_function(X_test)
        else:
            scores = model.predict(X_test).astype(float)

        info["mode"] = "supervised"

    elif model_name in UNSUPERVISED_MODELS:
        contamination = float(max(1e-4, min(0.49, y_train.mean() if y_train.mean() > 0 else 0.05)))

        model = make_unsupervised_model(model_name, contamination=contamination)
        model.fit(X_train)

        imputer = model.named_steps["imputer"]
        clf = model.named_steps["clf"]

        X_test_imp = imputer.transform(X_test)

        # IsolationForest: lower decision_function means more anomalous.
        raw = clf.decision_function(X_test_imp)
        scores = -raw

        info["mode"] = "unsupervised"
        info["contamination"] = contamination

    else:
        raise ValueError(model_name)

    return np.asarray(scores, dtype=float), info


def run_experiment(
    ds: Dict[str, Any],
    regime: str,
    ablation: str,
    model_name: str,
    force: bool = False,
) -> Dict[str, Any]:

    dataset_name = ds["name"]
    json_path, txt_path = experiment_paths(dataset_name, regime, ablation, model_name)

    if json_path.exists() and not force:
        print(f"[SKIP] {json_path.name}")
        return load_json(json_path)

    log_line(txt_path, "=" * 100)
    log_line(txt_path, f"RUN dataset={dataset_name}, regime={regime}, ablation={ablation}, model={model_name}")
    log_line(txt_path, "=" * 100)

    t0 = time.time()

    try:
        pert = build_perturbed_dataset_features(ds, regime, force=False)
        X, feature_names = get_ablation_matrix(pert, ablation)

        y = ds["y"].astype(int)
        train_idx = ds["train_idx"]
        val_idx = ds["val_idx"]
        test_idx = ds["test_idx"]

        # Use train+val for final fitting.
        train_full_idx = np.unique(np.concatenate([train_idx, val_idx]))

        log_line(txt_path, f"X shape: {X.shape}")
        log_line(txt_path, f"features: {len(feature_names)}")
        log_line(txt_path, f"n_edges after perturbation: {pert['n_edges']}")
        log_line(txt_path, f"train+val size: {len(train_full_idx)}")
        log_line(txt_path, f"test size: {len(test_idx)}")
        log_line(txt_path, f"total anomalies: {int(y.sum())}")
        log_line(txt_path, f"train anomalies: {int(y[train_full_idx].sum())}")
        log_line(txt_path, f"test anomalies: {int(y[test_idx].sum())}")

        scores, model_info = fit_predict_scores(
            model_name=model_name,
            X=X,
            y=y,
            train_idx=train_full_idx,
            test_idx=test_idx,
        )

        metrics = compute_metrics(y[test_idx], scores)
        runtime = time.time() - t0

        result = {
            "dataset": dataset_name,
            "regime": regime,
            "ablation": ablation,
            "model": model_name,
            "status": "ok",
            "n_nodes": int(len(y)),
            "n_edges_perturbed": int(pert["n_edges"]),
            "n_features": int(X.shape[1]),
            "n_train": int(len(train_full_idx)),
            "n_test": int(len(test_idx)),
            "n_anomalies_total": int(y.sum()),
            "n_anomalies_train": int(y[train_full_idx].sum()),
            "n_anomalies_test": int(y[test_idx].sum()),
            "feature_names": list(feature_names),
            "model_info": model_info,
            "metrics": metrics,
            "runtime_sec": float(runtime),
            "created_at": now_str(),
        }

        save_json(json_path, result)

        log_line(txt_path, "METRICS:")
        for k, v in metrics.items():
            log_line(txt_path, f"  {k}: {v}")

        log_line(txt_path, f"Runtime sec: {runtime:.2f}")
        log_line(txt_path, f"Saved JSON: {json_path}")

        return result

    except Exception as e:
        runtime = time.time() - t0

        result = {
            "dataset": dataset_name,
            "regime": regime,
            "ablation": ablation,
            "model": model_name,
            "status": "error",
            "error": repr(e),
            "runtime_sec": float(runtime),
            "created_at": now_str(),
        }

        save_json(json_path, result)
        log_line(txt_path, f"[ERROR] {repr(e)}")
        return result

In [12]:
# ============================================================
# Run all robustness experiments
# Resume-safe
# ============================================================

master_log = LOG_DIR / "MASTER_ROBUSTNESS_RUN_LOG.txt"

log_line(master_log, "=" * 120)
log_line(master_log, "START ROBUSTNESS / PERTURBATION EXPERIMENTS")
log_line(master_log, "=" * 120)

all_results = []

total = len(datasets) * len(PERTURBATION_REGIMES) * len(ABLATIONS) * len(ALL_MODELS)
counter = 0

print("Total planned experiments:", total)

for ds in datasets:
    for regime in PERTURBATION_REGIMES:
        for ablation in ABLATIONS:
            for model_name in ALL_MODELS:
                counter += 1

                print("\n" + "=" * 120)
                print(f"[{counter}/{total}] dataset={ds['name']} | regime={regime} | ablation={ablation} | model={model_name}")
                print("=" * 120)

                res = run_experiment(
                    ds=ds,
                    regime=regime,
                    ablation=ablation,
                    model_name=model_name,
                    force=False,
                )
                all_results.append(res)

log_line(master_log, "FINISHED ROBUSTNESS / PERTURBATION EXPERIMENTS")

[2026-05-17 14:15:40] ========================================================================================================================
[2026-05-17 14:15:40] START ROBUSTNESS / PERTURBATION EXPERIMENTS
[2026-05-17 14:15:40] ========================================================================================================================
Total planned experiments: 500

[1/500] dataset=FraudYelp | regime=original | ablation=attr_only | model=LogisticRegression
[2026-05-17 14:15:40] ====================================================================================================
[2026-05-17 14:15:40] RUN dataset=FraudYelp, regime=original, ablation=attr_only, model=LogisticRegression
[2026-05-17 14:15:40] ====================================================================================================
[2026-05-17 14:15:40] [SKIP] cached features exist: C:\Users\ivan\WORK\workshops\ICDM\Ablation\robustness_attribute_structure_outputs\cached_perturbed_features\FraudYelp__

In [13]:
# ============================================================
# Collect all JSON results
# ============================================================

def collect_results() -> pd.DataFrame:
    rows = []

    for path in sorted(JSON_DIR.glob("*.json")):
        try:
            r = load_json(path)
        except Exception as e:
            rows.append({
                "json_file": str(path),
                "status": "bad_json",
                "error": repr(e),
            })
            continue

        row = {
            "dataset": r.get("dataset"),
            "regime": r.get("regime"),
            "ablation": r.get("ablation"),
            "model": r.get("model"),
            "status": r.get("status"),
            "error": r.get("error"),
            "n_nodes": r.get("n_nodes"),
            "n_edges_perturbed": r.get("n_edges_perturbed"),
            "n_features": r.get("n_features"),
            "n_train": r.get("n_train"),
            "n_test": r.get("n_test"),
            "n_anomalies_total": r.get("n_anomalies_total"),
            "n_anomalies_train": r.get("n_anomalies_train"),
            "n_anomalies_test": r.get("n_anomalies_test"),
            "runtime_sec": r.get("runtime_sec"),
            "json_file": str(path),
        }

        metrics = r.get("metrics", {})
        for k, v in metrics.items():
            row[k] = v

        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        sort_cols = [c for c in ["dataset", "regime", "ablation", "model"] if c in df.columns]
        df = df.sort_values(sort_cols).reset_index(drop=True)

    return df


summary_df = collect_results()

summary_csv = SUMMARY_DIR / "robustness_results_summary.csv"
summary_xlsx = SUMMARY_DIR / "robustness_results_summary.xlsx"

summary_df.to_csv(summary_csv, index=False)

try:
    summary_df.to_excel(summary_xlsx, index=False)
except Exception as e:
    print("[WARN] Excel save failed:", repr(e))

print("Saved:", summary_csv)
print("Saved:", summary_xlsx)
display(summary_df)

[WARN] Excel save failed: ModuleNotFoundError("No module named 'openpyxl'")
Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\robustness_attribute_structure_outputs\summary\robustness_results_summary.csv
Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\robustness_attribute_structure_outputs\summary\robustness_results_summary.xlsx


,dataset,regime,ablation,model,status,error,n_nodes,n_edges_perturbed,n_features,n_train,...,k_at_5pct,hits_at_5pct,precision_at_k_at_5pct,recall_at_k_at_5pct,hits_at_k_at_5pct,k_at_10pct,hits_at_10pct,precision_at_k_at_10pct,recall_at_k_at_10pct,hits_at_k_at_10pct
0,FraudAmazon,attr_dropout_25,attr_only,ExtraTrees,ok,None,11944,9557648,25,6910,...,86,85,0.988372,0.461957,1.0,172,146,0.848837,0.793478,1.0
1,FraudAmazon,attr_dropout_25,attr_only,GradientBoosting,ok,None,11944,9557648,25,6910,...,86,83,0.965116,0.451087,1.0,172,154,0.895349,0.836957,1.0
2,FraudAmazon,attr_dropout_25,attr_only,IsolationForest,ok,None,11944,9557648,25,6910,...,86,42,0.488372,0.228261,1.0,172,72,0.418605,0.391304,1.0
3,FraudAmazon,attr_dropout_25,attr_only,LogisticRegression,ok,None,11944,9557648,25,6910,...,86,83,0.965116,0.451087,1.0,172,148,0.860465,0.804348,1.0
4,FraudAmazon,attr_dropout_25,attr_only,RandomForest,ok,None,11944,9557648,25,6910,...,86,83,0.965116,0.451087,1.0,172,153,0.889535,0.831522,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,FraudYelp,relation_dropout_50,struct_only,ExtraTrees,ok,None,45954,4023852,23,36762,...,459,124,0.270153,0.095385,1.0,919,234,0.254625,0.180000,1.0
496,FraudYelp,relation_dropout_50,struct_only,GradientBoosting,ok,None,45954,4023852,23,36762,...,459,155,0.337691,0.119231,1.0,919,282,0.306855,0.216923,1.0
497,FraudYelp,relation_dropout_50,struct_only,IsolationForest,ok,None,45954,4023852,23,36762,...,459,84,0.183007,0.064615,1.0,919,171,0.186072,0.131538,1.0
498,FraudYelp,relation_dropout_50,struct_only,LogisticRegression,ok,None,45954,4023852,23,36762,...,459,141,0.307190,0.108462,1.0,919,249,0.270947,0.191538,1.0


In [14]:
# ============================================================
# Completion and error diagnostics
# ============================================================

df = collect_results()

print("=" * 120)
print("STATUS COUNTS")
print("=" * 120)

display(df.groupby(["dataset", "status"]).size().reset_index(name="count"))

print("=" * 120)
print("COUNTS BY DATASET / REGIME")
print("=" * 120)

display(df.groupby(["dataset", "regime"]).size().reset_index(name="n_runs"))

print("=" * 120)
print("ERRORS")
print("=" * 120)

errors = df[df["status"] != "ok"].copy()

if errors.empty:
    print("No errors.")
else:
    display(errors[[
        "dataset", "regime", "ablation", "model", "status", "error", "json_file"
    ]])

STATUS COUNTS


,dataset,status,count
0,FraudAmazon,ok,250
1,FraudYelp,ok,250


COUNTS BY DATASET / REGIME


,dataset,regime,n_runs
0,FraudAmazon,attr_dropout_25,25
1,FraudAmazon,attr_dropout_50,25
2,FraudAmazon,attr_noise_10,25
3,FraudAmazon,attr_noise_25,25
4,FraudAmazon,attr_shuffle,25
5,FraudAmazon,degree_preserving_rewire,25
6,FraudAmazon,edges_removed,25
7,FraudAmazon,original,25
8,FraudAmazon,relation_dropout_25,25
9,FraudAmazon,relation_dropout_50,25


ERRORS
No errors.


In [15]:
# ============================================================
# Best result per dataset / regime / ablation
# ============================================================

df = collect_results()
ok = df[df["status"] == "ok"].copy()

best_by_roc = (
    ok
    .dropna(subset=["roc_auc"])
    .sort_values(["dataset", "regime", "roc_auc"], ascending=[True, True, False])
    .groupby(["dataset", "regime"])
    .head(10)
    .reset_index(drop=True)
)

best_csv = SUMMARY_DIR / "best_by_regime_roc_auc.csv"
best_by_roc.to_csv(best_csv, index=False)

print("Saved:", best_csv)

display(best_by_roc[[
    "dataset",
    "regime",
    "ablation",
    "model",
    "roc_auc",
    "pr_auc",
    "precision_at_k_at_pos",
    "recall_at_k_at_pos",
    "n_edges_perturbed",
    "n_features",
    "runtime_sec",
]])

Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\robustness_attribute_structure_outputs\summary\best_by_regime_roc_auc.csv


,dataset,regime,ablation,model,roc_auc,pr_auc,precision_at_k_at_pos,recall_at_k_at_pos,n_edges_perturbed,n_features,runtime_sec
0,FraudAmazon,attr_dropout_25,full,GradientBoosting,0.987532,0.929536,0.880435,0.880435,9557648,48,7.374999
1,FraudAmazon,attr_dropout_25,attr_only,GradientBoosting,0.986193,0.927598,0.869565,0.869565,9557648,25,1.847000
2,FraudAmazon,attr_dropout_25,full,RandomForest,0.977084,0.919661,0.864130,0.864130,9557648,48,1.707996
3,FraudAmazon,attr_dropout_25,full,LogisticRegression,0.975985,0.885041,0.847826,0.847826,9557648,48,0.100998
4,FraudAmazon,attr_dropout_25,attr_only,RandomForest,0.975909,0.914308,0.869565,0.869565,9557648,25,1.713022
...,...,...,...,...,...,...,...,...,...,...,...
195,FraudYelp,relation_dropout_50,attr_only,GradientBoosting,0.889330,0.667616,0.616923,0.616923,4023852,32,25.936998
196,FraudYelp,relation_dropout_50,full,LogisticRegression,0.830253,0.474300,0.473846,0.473846,4023852,55,1.097000
197,FraudYelp,relation_dropout_50,attr_only,LogisticRegression,0.777192,0.396541,0.410769,0.410769,4023852,32,0.415999
198,FraudYelp,relation_dropout_50,struct_only,GradientBoosting,0.715842,0.271654,0.285385,0.285385,4023852,23,25.372985


In [16]:
# ============================================================
# Best result per dataset / regime / ablation
# ============================================================

df = collect_results()
ok = df[df["status"] == "ok"].copy()

best_by_roc = (
    ok
    .dropna(subset=["roc_auc"])
    .sort_values(["dataset", "regime", "roc_auc"], ascending=[True, True, False])
    .groupby(["dataset", "regime"])
    .head(10)
    .reset_index(drop=True)
)

best_csv = SUMMARY_DIR / "best_by_regime_roc_auc.csv"
best_by_roc.to_csv(best_csv, index=False)

print("Saved:", best_csv)

display(best_by_roc[[
    "dataset",
    "regime",
    "ablation",
    "model",
    "roc_auc",
    "pr_auc",
    "precision_at_k_at_pos",
    "recall_at_k_at_pos",
    "n_edges_perturbed",
    "n_features",
    "runtime_sec",
]])

Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\robustness_attribute_structure_outputs\summary\best_by_regime_roc_auc.csv


,dataset,regime,ablation,model,roc_auc,pr_auc,precision_at_k_at_pos,recall_at_k_at_pos,n_edges_perturbed,n_features,runtime_sec
0,FraudAmazon,attr_dropout_25,full,GradientBoosting,0.987532,0.929536,0.880435,0.880435,9557648,48,7.374999
1,FraudAmazon,attr_dropout_25,attr_only,GradientBoosting,0.986193,0.927598,0.869565,0.869565,9557648,25,1.847000
2,FraudAmazon,attr_dropout_25,full,RandomForest,0.977084,0.919661,0.864130,0.864130,9557648,48,1.707996
3,FraudAmazon,attr_dropout_25,full,LogisticRegression,0.975985,0.885041,0.847826,0.847826,9557648,48,0.100998
4,FraudAmazon,attr_dropout_25,attr_only,RandomForest,0.975909,0.914308,0.869565,0.869565,9557648,25,1.713022
...,...,...,...,...,...,...,...,...,...,...,...
195,FraudYelp,relation_dropout_50,attr_only,GradientBoosting,0.889330,0.667616,0.616923,0.616923,4023852,32,25.936998
196,FraudYelp,relation_dropout_50,full,LogisticRegression,0.830253,0.474300,0.473846,0.473846,4023852,55,1.097000
197,FraudYelp,relation_dropout_50,attr_only,LogisticRegression,0.777192,0.396541,0.410769,0.410769,4023852,32,0.415999
198,FraudYelp,relation_dropout_50,struct_only,GradientBoosting,0.715842,0.271654,0.285385,0.285385,4023852,23,25.372985
